# Week 1 Assessment — Employee Data Analysis

Notebook companion for Task 3. Run cells from top to bottom after installing `requirements.txt`.

## 1. Imports and paths

In [ ]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd()
if not (ROOT / 'data').exists() and (ROOT.parent / 'data').exists():
    ROOT = ROOT.parent
DATA = ROOT / 'data'
REPORTS = ROOT / 'reports'
REPORTS.mkdir(exist_ok=True)
print('Project root:', ROOT)

## 2. NumPy marks array

In [ ]:
marks = np.array([[80, 75, 90], [70, 85, 68], [92, 88, 77], [60, 65, 70]])
weights = np.array([0.4, 0.35, 0.25])
print('Shape:', marks.shape)
print('Student averages:', marks.mean(axis=1))
print('Subject averages:', marks.mean(axis=0))
print('With grace marks:\n', marks + 5)
print('Weighted scores:', np.dot(marks, weights))

## 3. Inspect employee data with Pandas

In [ ]:
employees = pd.read_csv(DATA / 'employees.csv')
employees.info()
display(employees.describe(include='all'))
display(employees.loc[employees['city'] == 'Chennai'])
display(employees.iloc[:3])
display(employees[employees['salary'] > 50000])

## 4. Clean missing values, types, text and duplicates

In [ ]:
raw = pd.read_csv(DATA / 'employees_raw.csv')
before = {'rows': len(raw), 'missing_values': int(raw.isna().sum().sum()), 'duplicates': int(raw.duplicated().sum())}
raw['name'] = raw['name'].astype('string').str.strip().str.title()
raw['department'] = raw['department'].astype('string').str.strip().str.title()
raw['city'] = raw['city'].astype('string').str.strip().str.title()
raw['id'] = pd.to_numeric(raw['id'], errors='coerce').astype('Int64')
raw['salary'] = pd.to_numeric(raw['salary'], errors='coerce')
median_salary = raw['salary'].median()
raw['salary'] = raw['salary'].fillna(median_salary)
raw = raw.drop_duplicates().copy()
raw['salary'] = raw['salary'].astype(float)
after = {'rows': len(raw), 'missing_values': int(raw.isna().sum().sum()), 'duplicates': int(raw.duplicated().sum())}
summary = pd.DataFrame([{'stage':'Before', **before}, {'stage':'After', **after}])
display(summary)
print('Median used:', median_salary)
raw.to_csv(DATA / 'clean_data.csv', index=False)
display(raw)

## 5. Department aggregation and joins

In [ ]:
department_summary = raw.groupby('department').agg(headcount=('id','count'), average_salary=('salary','mean'), maximum_salary=('salary','max')).reset_index()
display(department_summary)
departments = pd.read_csv(DATA / 'departments.csv')
inner = raw.merge(departments, on='department', how='inner')
left = raw.merge(departments, on='department', how='left')
outer = raw.merge(departments, on='department', how='outer', indicator=True)
print('Inner rows:', len(inner), '| Left rows:', len(left))
print('Departments with no employees:', outer.loc[outer['_merge']=='right_only', 'department'].dropna().unique().tolist())

## 6. Charts

In [ ]:
avg_salary = employees.groupby('department')['salary'].mean().sort_index()
fig, ax = plt.subplots(figsize=(8,5))
bars = ax.bar(avg_salary.index, avg_salary.values)
ax.bar_label(bars, fmt='%.0f')
ax.set(title='Average Salary by Department', xlabel='Department', ylabel='Average salary')
ax.legend([bars], ['Average salary'])
fig.tight_layout()
fig.savefig(REPORTS / 'chart_bar_avg_salary.png', dpi=150)
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(8,5))
ax.hist(employees['salary'], bins=5, edgecolor='black', label='Employees')
ax.set(title='Employee Salary Distribution', xlabel='Salary', ylabel='Number of employees')
ax.legend(); fig.tight_layout(); fig.savefig(REPORTS / 'chart_histogram_salary.png', dpi=150); plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(8,5))
for department, subset in employees.groupby('department'):
    ax.scatter(subset['id'], subset['salary'], label=department)
ax.set(title='Employee ID vs Salary', xlabel='Employee ID', ylabel='Salary'); ax.legend(title='Department')
fig.tight_layout(); fig.savefig(REPORTS / 'chart_scatter_id_salary.png', dpi=150); plt.show()

In [ ]:
months = ['Jan','Feb','Mar','Apr','May','Jun']
sales = [12,15,14,18,22,25]
fig, ax = plt.subplots(figsize=(8,5))
ax.plot(months, sales, marker='o', label='Monthly sales')
ax.set(title='Monthly Sales (Jan–Jun)', xlabel='Month', ylabel='Sales'); ax.legend()
fig.tight_layout(); fig.savefig(REPORTS / 'chart_line_monthly_sales.png', dpi=150); plt.show()

## 7. Timing example (`%timeit`)
Run the following cell once as required by Task 4.

In [ ]:
%timeit employees['salary'].mean()